# Personalized Book Recommendation with CrewAI & Mem0

In this notebook we build a **personalized book recommendation system** by combining two powerful libraries:

| Library | Purpose |
|---------|---------|
| **CrewAI** | Orchestrates one or more AI *Agents* that collaborate on *Tasks* within a *Crew*. |
| **Mem0** | Provides a managed memory layer so the system can remember past interactions and improve recommendations over time. |

### What you will learn

1. How to define a CrewAI **Agent** with a specific role, backstory, and goal.
2. How to create a **Task** that the agent will execute.
3. How to wire an agent and task into a **Crew** with Mem0-backed memory.
4. How to **store conversation history** in Mem0 for long-term personalization.

### Prerequisites

| Requirement | Details |
|-------------|---------|
| Python | 3.10 + |
| API Keys | `OPENAI_API_KEY` (LLM) and `MEM0_API_KEY` (memory layer) stored in a `.env` file |
| Packages | `crewai`, `mem0ai`, `python-dotenv` |

> **Security note:** Never hard-code API keys in notebooks. We load them from environment variables via `python-dotenv`.


## 1. Import Required Libraries

We need four key imports:

- **`Agent`** — the autonomous entity that performs reasoning.
- **`Task`** — a unit of work assigned to an agent.
- **`Crew`** — the orchestrator that manages agents and tasks.
- **`Process`** — defines execution order (`sequential` or `hierarchical`).
- **`MemoryClient`** — Mem0's client for reading/writing long-term memory.
- **`dotenv`** — loads secrets from a `.env` file so they never appear in code.


In [1]:
# CrewAI core components for building multi-agent workflows
from crewai import Agent, Crew, Task, Process

# Mem0 client for persistent, user-scoped memory
from mem0 import MemoryClient

import os

# Load environment variables (.env file should contain OPENAI_API_KEY and MEM0_API_KEY)
from dotenv import load_dotenv
load_dotenv()

# Read the Mem0 API key once — reused by both the Crew memory config and the MemoryClient
MEM0_API_KEY = os.getenv("MEM0_API_KEY")

In [2]:
# ================================================================================
# Configure LLM
# ================================================================================

# Set up API key (uncomment and replace with your key if not using .env)
# os.environ['OPENAI_API_KEY'] = "YOUR_OPENAI_API_KEY"

# Initialize GPT-4 for advanced reasoning
# llm = LLM(model="gpt-4")
from helpers import get_llm

llm = get_llm(model = "groq/moonshotai/kimi-k2-instruct-0905")

LLM initialized: groq/moonshotai/kimi-k2-instruct-0905 (via databricks)


## 2. Define the Recommendation Function

This is the core of our system. Inside `get_recommendation()` we:

1. **Create an Agent** — give it a *role*, *backstory* (personality/context), and a *goal*.
2. **Create a Task** — describe what the agent should do and what output format we expect.
3. **Assemble a Crew** — bind the agent + task together, choose a `Process` (sequential here), and attach **Mem0 memory** so the crew can recall earlier interactions for the same user.
4. **Kick off** the crew and return the raw result.

> **How Mem0 memory works here:** When `memory=True` and a Mem0 config is provided, CrewAI automatically stores and retrieves relevant memories scoped to `user_id`. This means repeated calls for the same user will produce progressively better recommendations.


In [3]:
def get_recommendation(user_id: str, user_preferences: str) -> str:
    """Generate personalized book recommendations using a CrewAI agent backed by Mem0 memory."""

    # --- Step 1: Define the Agent ---
    # role  : short label shown in logs
    # backstory : gives the LLM persona-level context
    # goal  : high-level objective the agent tries to satisfy
    recommendation_agent = Agent(
        llm = llm,
        role="Book Recommendation",
        backstory="AI-powered book advisor who suggests books based on user preferences",
        goal="Provide personalised book recommendations.",
        verbose=True,
    )

    # --- Step 2: Define the Task ---
    # description  : the concrete instruction (user preferences are injected here)
    # expected_output : tells the agent what format the answer should take
    suggest_book_task = Task(
        description=(
            f"Suggest three personalised book recommendations based on "
            f"user preferences: {user_preferences}."
        ),
        expected_output="A list of three book titles with brief description",
        agent=recommendation_agent,
    )

    # --- Step 3: Assemble the Crew ---
    # memory=True  : enables long-term memory across runs
    # memory_config : points to Mem0 as the memory provider, scoped to user_id
    crew = Crew(
        agents=[recommendation_agent],
        tasks=[suggest_book_task],
        process=Process.sequential,
        memory=True,
        memory_config={
            "provider": "mem0",
            "config": {"user_id": user_id, "api_key": MEM0_API_KEY},
        },
    )

    # --- Step 4: Execute and return ---
    result = crew.kickoff()

    print("Final Recommendation:")
    print(result.raw)

    return result.raw

## 3. Initialize Memory Client & Define Storage Helper

Besides the *automatic* memory that CrewAI uses during execution, we also want to **explicitly store** the final recommendation as a conversation turn. This lets us:

- Build a browsable history of all recommendations per user.
- Feed richer context into future Crew runs.

Below we:
1. Instantiate the **`MemoryClient`** (using the same API key from `.env`).
2. Define a small helper — `store_user_recommendation()` — that writes a conversation list to Mem0.


In [4]:
# Initialize the Mem0 client (uses the same key loaded in cell 2)
client = MemoryClient(api_key=MEM0_API_KEY)


def store_user_recommendation(client: MemoryClient, user_id: str, conversation: list):
    """Persist a conversation (list of role/content dicts) to Mem0 under the given user_id."""
    client.add(conversation, user_id=user_id)


<!-- ## 4. Initialize Memory Client — MERGED into Section 3 above -->


## 4. Get User Input & Generate Recommendations

Time to run the system! We provide:

- **`user_id`** — a unique identifier so Mem0 can scope memories per user.
- **`user_preferences`** — a natural-language description of what the user likes.

The crew will spin up, the agent will reason about the preferences (and any prior memories for this user), and return three book suggestions.

> **Try it yourself:** Change `user_preferences` to something you enjoy — e.g., *"Mystery novels set in Victorian London"* — and re-run the cell.


In [5]:
# Unique user identifier — Mem0 uses this to store/retrieve memories per user
user_id = "Sourav"

# Natural-language description of what the user enjoys reading
user_preferences = "Science fiction books with time travel themes"

# Run the CrewAI pipeline and capture the recommendation text
result = get_recommendation(user_id, user_preferences)


ERROR:root:Error during short_term search: APIRemovedInV1.__init__() takes 1 positional argument but 2 were given
Traceback (most recent call last):
  File "/Users/sourav.banerjee/Documents/Codebases/AI_Engineering/AgenticAI_Projects_Demystified/.venv/lib/python3.12/site-packages/chromadb/api/models/CollectionCommon.py", line 90, in wrapper
    return func(self, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/sourav.banerjee/Documents/Codebases/AI_Engineering/AgenticAI_Projects_Demystified/.venv/lib/python3.12/site-packages/chromadb/api/models/CollectionCommon.py", line 301, in _validate_and_prepare_query_request
    request_embeddings = self._embed_record_set(record_set=query_records)
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/sourav.banerjee/Documents/Codebases/AI_Engineering/AgenticAI_Projects_Demystified/.venv/lib/python3.12/site-packages/chromadb/api/models/CollectionCommon.py", line 526, in _embed_record_set
   

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Book Recommendation                                                                                     │
│                                                                                                                 │
│  Task: Suggest three personalised book recommendations based on user preferences: Science fiction books with    │
│  time travel themes.                                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Book Recommendation                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  - The Time Machine – H. G. Wells (1895, 118 pp.; 3.89/5 Goodreads) The Victorian inventor who catapults to a   │
│  decaying future of gentle Eloi and subterranean Morlocks founded modern time-travel SF; if you liked Verne’s   │
│  or Stevenson’s pace and social edge, this is your next quick classic.                                          │
│  - Slaughterhouse-Five – Kurt Vonnegut (1969, 275 pp.; 4.1/5 Goodreads) “Unstuck in time” Billy Pilgrim         │
│  flickers between WWII Dresden and an alien zoo in a dark, anti-war satire; Vonnegut’s clipped irony will       │
│  appeal to fans of Catch-22 or Cat’s Cradle. Content: war trauma, brief profanity.                              │
│  - The Gone World – Tom Sweterlitsch (2018, 393 pp.; 3.94/5 Goodreads) A hardened NCIS agent uses wormhole      │
│  tech to jump into potential futures to stop a killer, only to uncover a timeline that ends in an inescapable   │
│  apocalypse; perfect for readers who liked the investigative noir of Blake Crouch’s Dark Matter or the cosmic   │
│  dread of The Peripheral. Content: graphic violence.                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ERROR:root:Error during short_term save: APIRemovedInV1.__init__() takes 1 positional argument but 2 were given
Traceback (most recent call last):
  File "/Users/sourav.banerjee/Documents/Codebases/AI_Engineering/AgenticAI_Projects_Demystified/.venv/lib/python3.12/site-packages/chromadb/api/models/CollectionCommon.py", line 90, in wrapper
    return func(self, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/sourav.banerjee/Documents/Codebases/AI_Engineering/AgenticAI_Projects_Demystified/.venv/lib/python3.12/site-packages/chromadb/api/models/CollectionCommon.py", line 406, in _validate_and_prepare_upsert_request
    upsert_embeddings = self._embed_record_set(record_set=upsert_records)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/sourav.banerjee/Documents/Codebases/AI_Engineering/AgenticAI_Projects_Demystified/.venv/lib/python3.12/site-packages/chromadb/api/models/CollectionCommon.py", line 526, in _embed_record_set
    

╭─────────────────────────────────────────── Trace Batch Finalization ────────────────────────────────────────────╮
│ ✅ Trace batch finalized with session ID: caf9bbf6-d579-4e77-9bf6-25566535cf3a                                  │
│                                                                                                                 │
│ 🔗 View here:                                                                                                   │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/caf9bbf6-d579-4e77-9bf6-25566535cf3a?access_code=TRA │
│ CE-56cfd638a2                                                                                                   │
│ 🔑 Access Code: TRACE-56cfd638a2                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Final Recommendation:
- The Time Machine – H. G. Wells (1895, 118 pp.; 3.89/5 Goodreads) The Victorian inventor who catapults to a decaying future of gentle Eloi and subterranean Morlocks founded modern time-travel SF; if you liked Verne’s or Stevenson’s pace and social edge, this is your next quick classic.  
- Slaughterhouse-Five – Kurt Vonnegut (1969, 275 pp.; 4.1/5 Goodreads) “Unstuck in time” Billy Pilgrim flickers between WWII Dresden and an alien zoo in a dark, anti-war satire; Vonnegut’s clipped irony will appeal to fans of Catch-22 or Cat’s Cradle. Content: war trauma, brief profanity.  
- The Gone World – Tom Sweterlitsch (2018, 393 pp.; 3.94/5 Goodreads) A hardened NCIS agent uses wormhole tech to jump into potential futures to stop a killer, only to uncover a timeline that ends in an inescapable apocalypse; perfect for readers who liked the investigative noir of Blake Crouch’s Dark Matter or the cosmic dread of The Peripheral. Content: graphic violence.


## 5. Store the Recommendation in Memory

Finally, we persist the interaction as a **conversation** (a list of `user` and `assistant` messages). This mirrors the chat format that Mem0 expects.

Storing recommendations explicitly gives us two benefits:

1. **Recall** — next time the same user asks, the agent can see what was already recommended and avoid duplicates.
2. **Audit trail** — we can later query Mem0 to review all recommendations made to a user.


In [6]:
# Build a conversation record in the standard role/content format that Mem0 expects
conversation = [
    {"role": "user", "content": f"Recommendation for {user_preferences}"},
    {"role": "assistant", "content": result},
]

# Persist the conversation to Mem0 under the same user_id
store_user_recommendation(client, user_id, conversation)
print("Recommendation stored successfully!")


Recommendation stored successfully!


In [7]:
get_recommendation("Sourav", "now suggest something different")

ERROR:root:Error during short_term search: APIRemovedInV1.__init__() takes 1 positional argument but 2 were given
Traceback (most recent call last):
  File "/Users/sourav.banerjee/Documents/Codebases/AI_Engineering/AgenticAI_Projects_Demystified/.venv/lib/python3.12/site-packages/chromadb/api/models/CollectionCommon.py", line 90, in wrapper
    return func(self, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/sourav.banerjee/Documents/Codebases/AI_Engineering/AgenticAI_Projects_Demystified/.venv/lib/python3.12/site-packages/chromadb/api/models/CollectionCommon.py", line 301, in _validate_and_prepare_query_request
    request_embeddings = self._embed_record_set(record_set=query_records)
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/sourav.banerjee/Documents/Codebases/AI_Engineering/AgenticAI_Projects_Demystified/.venv/lib/python3.12/site-packages/chromadb/api/models/CollectionCommon.py", line 526, in _embed_record_set
   

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Book Recommendation                                                                                     │
│                                                                                                                 │
│  Task: Suggest three personalised book recommendations based on user preferences: now suggest something         │
│  different.                                                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Book Recommendation                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  1. The Invisible Life of Addie LaRue – V. E. Schwab                                                            │
│     A 300-year curse renders Addie forgotten by everyone she meets, until one day a New York book-seller can    │
│  finally remember her.                                                                                          │
│     Rationale: You loved the “immortality with a cost” theme and lyrical, melancholic tone of The Song of       │
│  Achilles and On Earth We’re Briefly Gorgeous; Schwab explores those same aching themes but reshapes them into  │
│  a centuries-spanning, gender-flipped fantasy of self-determination rather than war.                            │
│                                                                                                                 │
│  2. The Long Way to a Small, Angry Planet – Becky Chambers                                                      │
│     A multispecies tunnelling ship takes a routine road-trip through the galaxy, turning into a warm,           │
│  slice-of-space exploration of found family and differing cultures.                                             │
│     Rationale: If you liked the intimate character vignettes and hopeful queer relationships in A Psalm for     │
│  the Wild-Built, Chambers offers that same gentle optimism but wrapped in episodic space-opera packaging.       │
│                                                                                                                 │
│  3. A Gentleman in Moscow – Amor Towles                                                                         │
│     Count Rostov is sentenced to lifelong house arrest inside Moscow’s grand Metropol Hotel in 1922, and        │
│  discovers that a life confined can still be infinitely wide.                                                   │
│     Rationale: While different in setting, it shares the reflective, character-driven cadence of A Man Called   │
│  Ove and the quiet celebration of everyday human connection rather than sweeping plots.                         │
│                                                                                                                 │
│  If you’d prefer something darker, shorter, or in a completely non-Western setting next time, just let me       │
│  know!                                                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ERROR:root:Error during short_term save: APIRemovedInV1.__init__() takes 1 positional argument but 2 were given
Traceback (most recent call last):
  File "/Users/sourav.banerjee/Documents/Codebases/AI_Engineering/AgenticAI_Projects_Demystified/.venv/lib/python3.12/site-packages/chromadb/api/models/CollectionCommon.py", line 90, in wrapper
    return func(self, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/sourav.banerjee/Documents/Codebases/AI_Engineering/AgenticAI_Projects_Demystified/.venv/lib/python3.12/site-packages/chromadb/api/models/CollectionCommon.py", line 406, in _validate_and_prepare_upsert_request
    upsert_embeddings = self._embed_record_set(record_set=upsert_records)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/sourav.banerjee/Documents/Codebases/AI_Engineering/AgenticAI_Projects_Demystified/.venv/lib/python3.12/site-packages/chromadb/api/models/CollectionCommon.py", line 526, in _embed_record_set
    

╭─────────────────────────────────────────── Trace Batch Finalization ────────────────────────────────────────────╮
│ ✅ Trace batch finalized with session ID: fdb2e275-fb40-47fb-a053-f48716e55ab2                                  │
│                                                                                                                 │
│ 🔗 View here:                                                                                                   │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/fdb2e275-fb40-47fb-a053-f48716e55ab2?access_code=TRA │
│ CE-9f3f28f02f                                                                                                   │
│ 🔑 Access Code: TRACE-9f3f28f02f                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Final Recommendation:
1. The Invisible Life of Addie LaRue – V. E. Schwab  
   A 300-year curse renders Addie forgotten by everyone she meets, until one day a New York book-seller can finally remember her.  
   Rationale: You loved the “immortality with a cost” theme and lyrical, melancholic tone of The Song of Achilles and On Earth We’re Briefly Gorgeous; Schwab explores those same aching themes but reshapes them into a centuries-spanning, gender-flipped fantasy of self-determination rather than war.

2. The Long Way to a Small, Angry Planet – Becky Chambers  
   A multispecies tunnelling ship takes a routine road-trip through the galaxy, turning into a warm, slice-of-space exploration of found family and differing cultures.  
   Rationale: If you liked the intimate character vignettes and hopeful queer relationships in A Psalm for the Wild-Built, Chambers offers that same gentle optimism but wrapped in episodic space-opera packaging.

3. A Gentleman in Moscow – Amor Towles  
   Count

'1. The Invisible Life of Addie LaRue – V. E. Schwab  \n   A 300-year curse renders Addie forgotten by everyone she meets, until one day a New York book-seller can finally remember her.  \n   Rationale: You loved the “immortality with a cost” theme and lyrical, melancholic tone of The Song of Achilles and On Earth We’re Briefly Gorgeous; Schwab explores those same aching themes but reshapes them into a centuries-spanning, gender-flipped fantasy of self-determination rather than war.\n\n2. The Long Way to a Small, Angry Planet – Becky Chambers  \n   A multispecies tunnelling ship takes a routine road-trip through the galaxy, turning into a warm, slice-of-space exploration of found family and differing cultures.  \n   Rationale: If you liked the intimate character vignettes and hopeful queer relationships in A Psalm for the Wild-Built, Chambers offers that same gentle optimism but wrapped in episodic space-opera packaging.\n\n3. A Gentleman in Moscow – Amor Towles  \n   Count Rostov is s